# 00 — The OpenAI API, seen once clearly

Some people in this room call the OpenAI API every week. Some have never sent a request. This notebook is for both. It is the smoke test for the course, and it is a walk through the one object every later module will sit on: a chat completion.

By the last guided cell you will have done four things: loaded the class key without printing it, made a normal call, streamed a call, and turned `usage` into dollars. Everything later — tools, the loop, MCP, frameworks — is more fields on the same response, plus two things this call does not have: memory, and hands.

The concept has already been introduced: credentials live outside the code, and every call has a price. This chapter makes the API itself visible.


## 1. Learn

You send a list of messages. The service returns one object. That is the whole Chat Completions API.

A message is a role plus content. Three roles matter:

- `system` — a standing instruction: tone, format, what not to do. The API does not store it. If you want it on the next call, you send it again.
- `user` — this turn's request.
- `assistant` — what the model said last time. You will send this back yourself once we have a loop.

The system message is how you steer without stuffing instructions into every user line. The first call uses one, so you can see it in the list.

```
notebook
    |
    |  load_dotenv, read MODEL_DEFAULT
    v
  .env  ----------------------->  OpenAI()
  (key, model, prices)                |
         never printed                |
                                      |  chat.completions.create(
                                      |      messages=[system, user]
                                      |  )
                                      v
                                ChatCompletion
                                   |  |  |
                                   |  |  +-- id, model, created
                                   |  |
                                   |  +-- choices[0].message.content
                                   |      choices[0].finish_reason
                                   |
                                   +-- usage.prompt_tokens
                                       usage.completion_tokens
                                              |
                                              v
                                         cost_usd
```

Set `stream=True` and the same request arrives as a sequence of small chunks. Each chunk has a `delta` — usually a few characters — instead of a finished `message`. We will add one idea at a time: print the words, save them, then ask for the bill. Streaming does not change what you pay. It changes when the words appear.

Two API details for this model family, before they waste a cell: use `max_completion_tokens`, not the older `max_tokens`. Do not pass `temperature`. Pin `reasoning_effort="none"` so a two-line riddle does not spend a hundred silent tokens thinking.

`MODEL_DEFAULT` is `gpt-5.4-nano`. `MODEL_STRONG` sits in `.env` for later. Both are read, never hard-coded.

One more fact, before any code: this API call has no memory and no tools. The model cannot see the previous cell unless we put that text back into `messages`. It cannot check the weather or look up a flight. It can only emit text. Those two gaps are the spine of the course. We will make you feel them in a moment, then leave them open on purpose.


## 2. Do

### Load the environment

Find `.env`, load it, and refuse to continue if the key or the model pin is missing. The search walks up from the current working directory so the notebook works whether Jupyter was started at the repo root or inside this folder.


In [1]:
from pathlib import Path
import os

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
price_in = float(os.environ["PRICE_INPUT_PER_MILLION"])
price_out = float(os.environ["PRICE_OUTPUT_PER_MILLION"])

assert api_key, "OPENAI_API_KEY is missing. Copy .env.example to .env and add the class key."
assert model, "MODEL_DEFAULT is missing from .env."

print("repo root:", ROOT)
print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)
print(f"prices: ${price_in}/1M input, ${price_out}/1M output")


repo root: /Users/tarekatwan/Downloads/ai_agents_course
OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano
prices: $0.2/1M input, $1.25/1M output


`OpenAI()` reads `OPENAI_API_KEY` from the environment on its own. You do not pass the key in. You do not print it.


### A first call

Two messages. The `system` line sets the format. The `user` line is the request. Both should be short enough to read on a projector.


In [2]:
client = OpenAI()

response = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "You write short riddles. Put the answer on its own last line.",
        },
        {"role": "user", "content": "Write a two-line riddle about a music shop."},
    ],
    max_completion_tokens=128,
    reasoning_effort="none",
)

print(response.choices[0].message.content)

In glass-lined halls I hum my wares to you,  
Six strings, bright keys—yet I sell quiet tunes.  
Answer: Music shop


### What came back

The riddle is the user message. The answer on its own last line is the system message doing its job — that layout was an instruction, not a coincidence.

`response` is not a string. It is a `ChatCompletion` object. The text you just printed is one field, several layers down. The rest of this cell names the fields you will keep reaching for.


In [3]:
choice = response.choices[0]
message = choice.message

print("id:            ", response.id)
print("object:        ", response.object)
print("model:         ", response.model)
print("created:       ", response.created)
print()
print("n choices:     ", len(response.choices))
print("finish_reason: ", choice.finish_reason)
print("message.role:  ", message.role)
print("message.content starts at:")
print(repr(message.content[:80]) if message.content else None)
print()
print("usage:         ", response.usage)

id:             chatcmpl-EELY8f2sd6EpTWvJCXxLCd6LugraE
object:         chat.completion
model:          gpt-5.4-nano-2026-03-17
created:        1787088136

n choices:      1
finish_reason:  stop
message.role:   assistant
message.content starts at:
'In glass-lined halls I hum my wares to you,  \nSix strings, bright keys—yet I sel'

usage:          CompletionUsage(completion_tokens=33, prompt_tokens=36, total_tokens=69, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))


A few things that dump is telling you:

- `model` is what the API actually ran. It may be a dated snapshot of `MODEL_DEFAULT`. That is expected.
- `choices` is a list because the API can return more than one completion (`n=2`, and so on). We always take `[0]`.
- `finish_reason` is `stop` when the model decided it was done, and `length` when it hit `max_completion_tokens`. If the riddle looks cut off, check this before you blame the prompt.
- `message.role` is `assistant`. When we build a loop, this whole message goes back on the list as the next turn.
- `message.content` is the text. Later, when the model wants a tool, the interesting field on this same object is `tool_calls`, and `content` may be empty. That is module 02. The object does not change shape.

The raw dump is worth seeing once, so the SDK stops feeling like a black box.


In [4]:
response.model_dump()

{'id': 'chatcmpl-EELY8f2sd6EpTWvJCXxLCd6LugraE',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'logprobs': None,
   'message': {'content': 'In glass-lined halls I hum my wares to you,  \nSix strings, bright keys—yet I sell quiet tunes.  \nAnswer: Music shop',
    'refusal': None,
    'role': 'assistant',
    'annotations': [],
    'audio': None,
    'function_call': None,
    'tool_calls': None}}],
 'created': 1787088136,
 'model': 'gpt-5.4-nano-2026-03-17',
 'object': 'chat.completion',
 'metadata': None,
 'moderation': None,
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': {'completion_tokens': 33,
  'prompt_tokens': 36,
  'total_tokens': 69,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0,
   'text_tokens': None},
  'prompt_tokens_details': {'audio_tokens': 0,
   'cache_write_tokens': None,
   'cached_tokens': 0,
   'image_tokens': None,
   'text_tokens': 

### What this call cannot do

Two gaps. Both are easy to believe as slides and hard to forget once you have seen them.

**No memory.** The riddle lives in the `response` object in this kernel. It does not live in the model. A new `create()` with a new `messages` list is a stranger — the system message is gone too, unless we send it again. We will ask what the riddle was, and we will not send the riddle back.


In [5]:
forgotten = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": "What riddle did you just tell me? Quote it."}
    ],
    max_completion_tokens=128,
    reasoning_effort="none",
)
print(forgotten.choices[0].message.content)

I didn’t tell you any riddle in this chat. There isn’t a riddle for me to quote.


It does not know. The fix is not mysterious: you keep the list. Next turn you send `system`, `user`, `assistant`, `user` again. That list *is* short-term memory. Module 03 builds the loop that appends to it. Module 05 is what you do when the list gets expensive.

**No tools.** The model also has no network, no clock, and no live data. It cannot check the weather in Prague, and it cannot look up whether a flight is delayed. Ask anyway. Watch whether it admits that or invents a number.


In [6]:
invented = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "user", "content": "What is the weather in Prague right now? Give the temperature in Celsius."}
    ],
    max_completion_tokens=64,
    reasoning_effort="none",
)
print(invented.choices[0].message.content)

I can’t access live weather data from here, so I can’t tell the current temperature in Prague “right now.”  

If you tell me what weather source/app you’re using (or share a screenshot/link), I can help you read the temperature in °C. Otherwise, you can check quickly via


Whatever temperature you just got is not from a weather station. The model has no network. It completed a plausible sentence. A flight-status question would have gone the same way: a confident arrival time, and no airline API behind it.

The two diagrams from the next technical module, early:

```
assumed     user --> software --> LLM --> tools

actual      user --> software --> LLM
                              |
                              +------> tools   (your code decides)
```

Module 02 is that picture. Module 03 is the loop that carries memory. We are not building any of that today. We are making sure you will want it.


### The same call, streamed

`stream=True` returns an iterator, not a finished `ChatCompletion`. Each item is a chunk. The text lives on `chunk.choices[0].delta.content` — a few characters, not the whole answer.

Some chunks have no text (`delta.content` is `None`). Check before you print. That is the only new idea in this cell.


In [7]:
stream = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "You write short riddles. Put the answer on its own last line.",
        },
        {"role": "user", "content": "Write a two-line riddle about a music shop."},
    ],
    max_completion_tokens=128,
    reasoning_effort="none",
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)


I

’m

 where

 melodies

 are

...

 years

.

A

 music

 shop

The words went to the screen and then they were gone. We do not have the riddle as a string. Same stream again, and this time we append each piece as it arrives.


In [8]:
stream = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "You write short riddles. Put the answer on its own last line.",
        },
        {"role": "user", "content": "Write a two-line riddle about a music shop."},
    ],
    max_completion_tokens=128,
    reasoning_effort="none",
    stream=True,
)

text = ""
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)
        text = text + chunk.choices[0].delta.content

print()
print()
print(text)


I

 keep

 melodies

 in

 rows

...

 around

?

Music

 shop



I keep melodies in rows, where every chord can be found,  
Strum my sign to find your match—what shop am I around?  

Music shop


Now we have the string. We still do not have the bill. A bare stream has no `usage` — that field is `None` unless we ask.

One extra argument: `stream_options={"include_usage": True}`. The last chunk then carries the same `usage` object as a normal call. The last chunk is often text-free, so check `chunk.choices` before you read `delta`.


In [9]:
stream = client.chat.completions.create(
    model=model,
    messages=[
        {
            "role": "system",
            "content": "You write short riddles. Put the answer on its own last line.",
        },
        {"role": "user", "content": "Write a two-line riddle about a music shop."},
    ],
    max_completion_tokens=128,
    reasoning_effort="none",
    stream=True,
    stream_options={"include_usage": True},
)

text = ""
usage = None
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)
        text = text + chunk.choices[0].delta.content
    if chunk.usage:
        usage = chunk.usage

print()
print()
print(usage)


In

 glass

 and

 velvet

 halls

...

:

 A

 music

 shop



CompletionUsage(completion_tokens=35, prompt_tokens=36, total_tokens=71, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))


Streaming does not make the call cheaper. You pay for the same tokens. You use it when a person is watching — a projector, a chat window — and waiting for the whole object would feel like a hang.

If you need the text and the bill, this last form is the one to keep.


## 3. Observe

Go back to the first, non-streamed `response`. The useful object for money is `usage`. Print it. Then turn tokens into dollars with the prices from `.env`.

On this model family, reasoning tokens are billed as output even when they never appear on the screen. They show up inside `usage` if they happened.


In [10]:
usage = response.usage
print(usage)

cost = usage.prompt_tokens / 1_000_000 * price_in + usage.completion_tokens / 1_000_000 * price_out
print(f"this call:            ${cost:.6f}")
print(f"1,000 of this call:   ${1000 * cost:.4f}")
print(f"100,000 of this call: ${100000 * cost:.2f}")


CompletionUsage(completion_tokens=33, prompt_tokens=36, total_tokens=69, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))
this call:            $0.000048
1,000 of this call:   $0.0485
100,000 of this call: $4.84


One riddle is a fraction of a cent. That is not the lesson. The lesson is the multiplier. An app that makes this call on every page load, or a company whose agent loop retries and fans out to thousands of requests a day, is spending `usage` in a tight loop. The same object you just printed is how you will know whether a design is cheap or a bill, before finance finds out for you.

If `content` is empty and `finish_reason` is `length`, the completion budget was spent on reasoning and nothing was left for words. Raise `max_completion_tokens` or keep `reasoning_effort="none"`. If the call raises an authentication error, the key is wrong or not loaded. If it raises a model-not-found error, `MODEL_DEFAULT` in `.env` does not match a model this key can see.


## 4. Challenge

Stream a one-sentence joke about invoices. Use the **second** stream — the one with `stream_options={"include_usage": True}` — so you get the words and the bill.

Same settings as the guided cells: `model`, `max_completion_tokens=128`, `reasoning_effort="none"`.

When you are done you should have:

- `text` — the assembled joke, a non-empty string
- `prompt_tokens` — an integer greater than zero
- `cost` — dollars, greater than zero

Print the joke and the cost. The next cell checks those three names. Do not edit it to make it pass.


In [ ]:
# Stream a one-sentence joke about invoices.
# Bind text, prompt_tokens, and cost.


In [ ]:
assert text and text.strip(), "text should be the assembled joke"
assert prompt_tokens > 0, "prompt_tokens should come from usage"
assert cost > 0, "cost should be dollars, greater than zero"
print(text)
print(f"prompt_tokens={prompt_tokens}  cost=${cost:.6f}")
